In [1]:
import pandas as pd

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# DATASET

In [2]:
data = pd.read_csv("/kaggle/input/datasets/filippotenani/uniting-dataset/UNITING_Dataset.csv") 

In [4]:
data.head()

,Creator name,Creator_gender,Filename,Social permalink,Channel,Followers,Type of content,Post creation date,Post creation time,Post caption,Reach,Likes,Comments,Total clicks,Brand name,Industry,Local,Brand_SM,media_duration_sec,face_frame_ratio,first_face_position_ratio,motion_level,saturation,luminance,contrast,sharpness,color_complexity
0,Guido Milani,m,149_960_2230,NaN,instagram.com/guidomilani,122281,INSTAGRAM_STORY,2024-05-15,17:23:02,NaN,553,NaN,NaN,13.0,Prada,Lifestyle,1,34000000,2.566667,1.0,0.0,0.949434,0.545074,114.898736,48.760628,103.385946,0.448271
1,Ilaria Limelli,f,501_1094_8748,NaN,instagram.com/ila_limelli,168801,INSTAGRAM_STORY,2025-09-20,20:32:40,NaN,15000,NaN,NaN,NaN,iliad,Services,0,137000,3.033333,0.0,NaN,0.924965,0.321415,84.088746,50.524906,1451.237177,0.797086
2,Edo e Marti,o,469_1840_8098,NaN,instagram.com/edoardoandmartina,361436,INSTAGRAM_STORY,2025-07-02,18:14:01,NaN,42712,NaN,NaN,403.0,Decathlon,Retail,0,313000,3.041667,0.0,NaN,0.000394,0.744234,26.504795,56.794075,1468.042620,0.552016
3,Sistiana Lombardi,f,125_757_2727,NaN,instagram.com/sistiana,83703,INSTAGRAM_STORY,2024-06-13,12:13:38,NaN,8000,NaN,NaN,NaN,SUPRADYN,Health,0,4000,3.100000,0.0,NaN,0.282641,0.210730,127.444795,69.210020,53.633243,0.708601
4,Eleonora Gardelli,f,526_1999_9290,NaN,instagram.com/eleonora_gardelli_,22985,INSTAGRAM_STORY,2025-10-30,21:45:28,NaN,4173,NaN,NaN,6.0,Decathlon,Retail,0,313000,3.583333,1.0,0.0,0.819924,0.334941,76.329992,44.254684,352.039550,0.640383


In [5]:
data.shape

(3243, 27)

In [ ]:
data.columns

# DATA CLEANING

For each column we check for NaN values and any potential issues

## CREATOR NAME

In [ ]:
data["Creator name"].head(100)
# name of the content creator

In [5]:
data["Creator name"].isna().sum() 
# no NaN

np.int64(0)

In [ ]:
data["Creator name"].value_counts() 
# no typos, no errors found

## CREATOR GENDER

In [ ]:
data["Creator_gender"].head(100)
# gender of the content creator: male 'm', female 'f', group 'o'

In [6]:
data["Creator_gender"].isna().sum() 
# no NaN

np.int64(0)

In [ ]:
data["Creator_gender"].value_counts() 
# male 'm', female 'f', group 'o'

In [ ]:
data[data["Creator_gender" ] == "o"]["Creator name"].value_counts()
# let's check the groups

## FILENAME

In [ ]:
data["Filename"].head(100)
# unique id that identifies the provided video files so we can match this dataset to the corresponding videos

In [ ]:
data["Filename"].isna().sum()
# no NaN

In [ ]:
data["Filename"].value_counts() 
# all different, correct

## SOCIAL PERMALINK

In [ ]:
data["Social permalink"].head(100)
# link to the video (only for reels and tiktok)

In [ ]:
data["Social permalink"].isna().sum() 
# 2230 NaN out of 3243, correct because they are only reels/tiktok

In [ ]:
data["Social permalink"].value_counts() 
# each social permalink appears only once except in a few cases where it appears twice, meaning the same video is present twice
# let's check if it's an error or correct
counts = data["Social permalink"].value_counts()
duplicates = counts[counts == 2].index
print(data[data["Social permalink"].isin(duplicates)][["Social permalink", "Creator name"]])
# we see that the repeated social permalinks (i.e. same video) are collaborations between two different creators

## CHANNEL

In [ ]:
data["Channel"].head(100)
# link to the creator's channel

In [ ]:
data["Channel"].isna().sum() 
# no NaN

In [ ]:
data["Channel"].value_counts() 
# no issues

## FOLLOWERS

In [ ]:
data["Followers"].head(100)
# number of followers of the content creator

In [ ]:
data["Followers"].isna().sum() 
# no NaN

In [ ]:
follow = data[["Followers", "Channel"]].drop_duplicates().sort_values("Channel")
follow.duplicated().sum()
# all channels have consistent follower counts, i.e. there's no channel with e.g. 100 followers in one row and 150 in another

In [ ]:
data[["Channel", "Followers"]].drop_duplicates(subset="Channel").sort_values("Followers")
# all follower numbers are consistent
# only sanross93 in the dataset has few on tiktok (190), checking directly on her tiktok profile she indeed has few
# but looking her up on instagram she's a famous model so it's fine, she just has few followers on tiktok

## TYPE OF CONTENT

In [ ]:
data["Type of content"].head(100)
# type of content: instagram story, instagram reel, tiktok post

In [ ]:
data["Type of content"].isna().sum() 
# no NaN

In [ ]:
data["Type of content"].value_counts() 
# no issues

## POST CREATION DATE

In [ ]:
data["Post creation date"].head(100)
# post creation date

In [ ]:
data["Post creation date"].isna().sum() 
# no NaN

In [ ]:
data["Post creation date"].value_counts() 
# no issues

## POST CREATION TIME

In [ ]:
data["Post creation time"].head(100)
# post creation time

In [ ]:
data["Post creation time"].isna().sum() 
# no NaN

In [ ]:
data["Post creation time"].value_counts() 
# most rows have time 00:00:00 which effectively represents a NaN
# and those at 01:00:00 and 02:00:00, are they also NaN??? the professor said no, they should be kept as they are
# others have a round time like 15:00:00 which we think represents an approximate time (posted around 15:00:00)
# others instead have a precise time with hours, minutes and seconds

## POST CAPTION

In [ ]:
data["Post caption"].head(100)
# these are the post captions (only for reels and tiktok)
# these will be analyzed with LLM

In [ ]:
data["Post caption"].isna().sum() # 2230 nan su 3243 
data[data["Post caption"].notna() & data["Social permalink"].notna()].shape[0]
# the non-NaN values are all and only those that also have the permalink, so the NaN values are the stories which is correct

## REACH

In [ ]:
data["Reach"].head(100)
# post reach: number of unique people who saw the content

In [ ]:
data["Reach"].isna().sum() 
# no NaN

In [ ]:
data["Reach"].value_counts().sort_index() 
# no issues

## ! LIKES

In [ ]:
data["Likes"].head(100)
# number of likes of the content (only for reels and tiktok)

In [8]:
# likes data is stored as strings, we convert them to integers
data["Likes"].dtypes
data["Likes"].apply(type).value_counts()
data["Likes"] = pd.to_numeric(data["Likes"], errors='coerce').astype('Int64')

In [9]:
data["Likes"].isna().sum() 
# 2281 NaN out of 3243, so 2230 NaN are the stories, while 51 NaN are actual missing values to impute

# since there are only 51 we go check the posts one by one
data[(data["Type of content"] == "INSTAGRAM_REEL") | (data["Type of content"] == "TIKTOK_POST")]["Likes"].isna().sum() # 51
# let's check the posts one by one
mask = (data["Type of content"] == "INSTAGRAM_REEL") | (data["Type of content"] == "TIKTOK_POST")
print(data[mask & data["Likes"].isna()][["Likes", "Social permalink"]])
# PROBLEM: for these posts instagram does not show the number of likes

      Likes                            Social permalink
113    <NA>  https://www.instagram.com/reel/DRJ7bGODIvq
126    <NA>  https://www.instagram.com/reel/DQwL7aEjErd
193    <NA>  https://www.instagram.com/reel/DQO1Yi7DNH0
930    <NA>  https://www.instagram.com/reel/C8t3OLdskDL
1225   <NA>  https://www.instagram.com/reel/DQKJxacjcC6
1427   <NA>  https://www.instagram.com/reel/DMFRHrLqNwM
1555   <NA>  https://www.instagram.com/reel/C9HffxMN7tm
1749   <NA>  https://www.instagram.com/reel/DDPl66xo2B5
1878   <NA>  https://www.instagram.com/reel/DQT_DwUDT0X
2044   <NA>  https://www.instagram.com/reel/C7E3KaPK9lW
2101   <NA>  https://www.instagram.com/reel/DQCGlvDjOzV
2189   <NA>  https://www.instagram.com/reel/C9SIuPdIgEx
2208   <NA>  https://www.instagram.com/reel/DQrrU2GjOhG
2243   <NA>  https://www.instagram.com/reel/DGIaowSuO37
2277   <NA>  https://www.instagram.com/reel/DQKLouBDMpx
2278   <NA>  https://www.instagram.com/reel/DQKLouBDMpx
2317   <NA>  https://www.instagram.com/reel/C56C

In [11]:
# so we find an alternative to fill in the missing likes
# we estimate the likes based on comments:
# for each post with missing likes, we look at what percentile its number of comments falls in the comments distribution of the dataset,
# then we assign the likes value (to that post with missing likes) that falls at that same percentile in the likes distribution

# select only REEL and TIKTOK
mask = (data["Type of content"] == "INSTAGRAM_REEL") | (data["Type of content"] == "TIKTOK_POST")

# we take only reel/tiktok with non-null Likes (used to compute quantiles)
valid = data[mask & data["Likes"].notna()]

def impute_likes(row, valid_data):
    # computes at what quantile this post's number of comments falls
    # relative to the comments distribution in the valid subset
    quantile = (valid_data["Comments"] < row["Comments"]).mean()
    # returns the Likes value corresponding to that quantile
    # we round the likes value to integer because likes are integers
    return int(round(valid_data["Likes"].quantile(quantile)))
    
# select only the rows to impute: only reel/tiktok with null Likes
mask_null = mask & data["Likes"].isna()
data.loc[mask_null, "Likes"] = data[mask_null].apply(
    lambda row: impute_likes(row, valid), axis=1
)

In [12]:
# let's verify there are only 2230 NaN i.e. only stories
data["Likes"].isna().sum() # 2230, quindi solo stories

# let's visualize the filled values (they had these indices)
indices = [113, 126, 193, 930, 1225, 1427, 1555, 1749, 1878, 2044, 2101, 2189, 2208, 2243, 2277, 2278, 2317, 2318, 2339, 2389, 2400, 2415, 
           2432, 2441, 2442, 2472, 2602, 2646, 2661, 2669, 2686, 2752, 2765, 2841, 2845, 2876, 2890, 2939, 2943, 2963, 3012, 3019, 3071, 
           3102, 3103, 3125, 3130, 3150, 3167, 3197, 3224]
print(data.loc[indices, ["Likes", "Comments"]])

      Likes  Comments
113    3014      33.0
126    3453      39.0
193       5       0.0
930    3453      39.0
1225    256       5.0
1427   2472      28.0
1555    769      12.0
1749   4147      46.0
1878   1277      19.0
2044   1516      21.0
2101    700      11.0
2189   1023      16.0
2208    296       6.0
2243    455       8.0
2277   7154      78.0
2278   7154      78.0
2317   5497      59.0
2318   5497      59.0
2339    296       6.0
2389    938      15.0
2400    635      10.0
2415   1936      25.0
2432    381       7.0
2441   2336      27.0
2442    516       9.0
2472   6136      65.0
2602   4966      56.0
2646   1023      16.0
2661   6212      67.0
2669   1398      20.0
2686    938      15.0
2752   1850      24.0
2765    296       6.0
2841   2472      28.0
2845    700      11.0
2876   2472      28.0
2890   4209      47.0
2939   1516      21.0
2943    887      14.0
2963  13558     135.0
3012    938      15.0
3019   1689      23.0
3071    455       8.0
3102   3241      36.0
3103   324

In [ ]:
# let's look at the unique likes values in the dataset
data[["Likes", "Social permalink"]].drop_duplicates(subset="Likes").sort_values("Likes")
# there are some videos with very few likes (5, 12, 15, ...), we checked those videos on tiktok and indeed the likes are few
# but the creators have many followers so we think they are videos that performed poorly

## COMMENTS

In [ ]:
data["Comments"].head(100)
# number of comments of the content (only for reels and tiktok)

In [ ]:
data["Comments"].isna().sum() 
# 2230 NaN out of 3243

In [ ]:
data[["Comments", "Social permalink"]].drop_duplicates(subset="Comments").sort_values("Comments")
# there are some videos with very few comments (0, 1, 2, ...), we checked those videos on tiktok and indeed the comments are few
# but the creators have many followers so we think they are videos that performed poorly

## ! TOTAL CLICKS

In [ ]:
data["Total clicks"].head(100)
# number of clicks on sponsored links (only for stories)

In [14]:
data["Total clicks"].isna().sum() 
# 1280 NaN out of 3243

# let's check the NaN values specific to stories, since reel/tiktok don't have clicks
data[data["Type of content"] == "INSTAGRAM_STORY"]["Total clicks"].isna().sum()
# 267 missing values
# so let's check if they are non-sponsored stories (no brand name) or if they are just NaN values
data["Brand name"].isna().sum() 
# brand name has no missing values so all stories are sponsored so the 267 are NaN values to fill

np.int64(0)

In [16]:
# we save in a vector the indices of the NaN values of stories so that after filling them we can verify they were filled correctly
missing_clicks_idx = data[
    (data["Type of content"] == "INSTAGRAM_STORY") & (data["Total clicks"].isna())
].index.tolist()

In [21]:
# To impute the missing Total clicks in Instagram Stories:
# 1) We compute the Total clicks / Reach ratio for each story with complete data
# 2) We average this ratio per creator (how many clicks each unit of reach generates on average for that creator)
# 3) For stories with missing Total clicks, we estimate the value as: story Reach * average ratio of its creator

# select only stories
stories = data[data["Type of content"] == "INSTAGRAM_STORY"].copy()

# average Total_clicks/reach per creator
creator_ratio = (
    stories[stories["Total clicks"].notna()]
    .groupby("Creator name")
    .apply(lambda x: (x["Total clicks"] / x["Reach"]).mean(), include_groups=False)
)

# impute the missing values
def impute_clicks(row):
    if pd.isna(row["Total clicks"]):
        ratio = creator_ratio.get(row["Creator name"], creator_ratio.mean())
        return round(row["Reach"] * ratio)
    return row["Total clicks"]

stories["Total clicks"] = stories.apply(impute_clicks, axis=1)

# put the imputed values back into the original dataframe
data.loc[stories.index, "Total clicks"] = stories["Total clicks"]

# visualizziamo i valori riempiti
print(data.loc[missing_clicks_idx, ["Total clicks", "Reach"]])

      Total clicks   Reach
1             25.0   15000
3             66.0    8000
9             15.0    8000
13            75.0    9000
16            66.0    8000
18            33.0    4000
20           233.0   28829
24            54.0    6465
27            36.0   80000
31            41.0    5000
34            62.0    6000
35           124.0   15000
41            50.0    6000
44             7.0     728
52            41.0    5000
53           680.0   42447
57            21.0    2500
64            66.0    8000
66           298.0   36000
72             4.0     640
78           207.0   25000
80            29.0    3500
85           204.0   35000
87            50.0    6000
93           124.0   15000
96            29.0    3500
97             3.0    2496
101           10.0    1098
109            7.0     897
116            3.0     577
121            6.0    1480
157           37.0    4500
160           17.0    3000
170           99.0   12000
172           41.0    5000
173           13.0    4714
1

In [22]:
# let's verify that all NaN values for stories have been filled
data["Total clicks"].isna().sum()
# 1013 NaN which are exactly the reels/tiktok ones, correct

np.int64(1013)

In [ ]:
# let's look at the unique Total clicks values
data["Total clicks"].drop_duplicates().sort_values()
# here too, like for the previous columns, there are some very low values but we cannot verify them since the video link is missing
# based on what we saw with the previous columns we think this is normal
# no other issues found

## BRAND NAME

In [ ]:
data["Brand name"].head(100)
# name of the brand sponsoring the video

In [ ]:
data["Brand name"].isna().sum() 
# no NaN

In [5]:
data["Brand name"].value_counts()
# there are multiple Kellogg's (Extra, Barrette, Special K), should we group them under Kellogg's?
# there are multiple Esselunga (Esselunga, Essebella i.e. the perfumery of Esselunga), should we group them under Esselunga?
# no other issues

Brand name
Vileda                    448
AMAZON                    365
iliad                     301
Decathlon                 289
SUPRADYN                  175
Esselunga                 155
Shaka Beauty              128
Mulino Bianco              99
PLOOM                      93
BENNET                     88
Dietor                     70
Metagenics                 69
Transitions                67
BIRRA MORETTI              63
Kellogg's Extra            63
Vogue Eyewear              57
Generali                   49
Sapporo                    45
Happy 2O                   42
Hoover                     42
Prada                      35
Ichnusa                    34
Be Charge - Plenitude      32
Heineken                   31
Alfa Romeo                 31
MC ARTHUR GLEN             28
WELEDA                     28
Birra Messina              27
Weetabix                   24
Galbani                    24
Dietorelle                 21
Tigullio                   20
Tequila Patron             19

## INDUSTRY

In [ ]:
data["Industry"].head(100)
# industry in which the "Brand name" (i.e. the video sponsor) operates

In [ ]:
data["Industry"].isna().sum() 
# no NaN

In [ ]:
data["Industry"].value_counts()
data.groupby("Brand name")["Industry"].unique()
# the same "Brand name" always corresponds to the same "Industry" value so there are no issues of this kind
# also the brands seem to have the correct "Industry" value for the sector they operate in

## LOCAL

In [ ]:
data["Local"].head(100)
# binary variable indicating whether "Brand name" (i.e. the video sponsor) is Italian or not

In [ ]:
data["Local"].isna().sum() 
# no NaN

In [ ]:
data["Local"].value_counts()
data.groupby("Brand name")["Local"].unique()
# the same "Brand name" always corresponds to the same "Local" value so there are no issues of this kind
# also Italian brands correctly have "Local" = 1

## BRAND SM

In [ ]:
data["Brand_SM"].head(100)
# number of social media followers of the "Brand name" at the time of the creator's video sponsorship

In [ ]:
data["Brand_SM"].isna().sum() 
# no NaN

In [ ]:
data["Brand_SM"].drop_duplicates().sort_values()
# no issues

## MEDIA DURATION SEC

In [ ]:
data["media_duration_sec"].head(100)
# video duration in seconds

In [ ]:
data["media_duration_sec"].isna().sum() 
# no NaN

In [ ]:
data["media_duration_sec"].drop_duplicates().sort_values()
data["media_duration_sec"].describe()
# no issues

## FACE FRAME RATIO

In [ ]:
data["face_frame_ratio"].head(100)
# percentage of video frames containing a human face

In [ ]:
data["face_frame_ratio"].isna().sum()
# no NaN

In [ ]:
data["face_frame_ratio"].drop_duplicates().sort_values()
data["face_frame_ratio"].describe()
# it's a percentage and correctly ranges from 0.0 to 1.0
# no issues

In [ ]:
data["face_frame_ratio"].value_counts()
# 147 videos without a face i.e. with face_frame_ratio = 0

## ! FIRST FACE POSITION RATIO

In [ ]:
data["first_face_position_ratio"].head(100)
# the percentage position of the first video frame showing a face

In [22]:
data["first_face_position_ratio"].isna().sum()
data[data["face_frame_ratio"] == 0]["first_face_position_ratio"].isna().all()
# 147 NaN which are the ones without a face

# since we can't have NaN values in the model, we must fill the NaN values of first_face_position_ratio
# we can't fill with 0 because that would mean the face appears immediately in the first frame of the video
# so we fill the missing values with 1, i.e. we say the face appears exactly in the last frame of the video
# we think this is a good proxy for videos without a face
# in any case in the model we will have a categorical variable face/noface
data["first_face_position_ratio"] = data["first_face_position_ratio"].fillna(1)
data["first_face_position_ratio"].isna().sum()

np.int64(0)

In [23]:
data["first_face_position_ratio"].drop_duplicates().sort_values()
data["first_face_position_ratio"].describe()
# no issues

count    3243.000000
mean        0.080958
std         0.229305
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: first_face_position_ratio, dtype: float64

## MOTION LEVEL

In [ ]:
data["motion_level"].head(100)
# level of motion in the video, referring to movements of the subject and the camera

In [ ]:
data["motion_level"].isna().sum()
# no NaN

In [ ]:
data["motion_level"].drop_duplicates().sort_values()
data["motion_level"].describe()
# there don't seem to be any issues
# however there are rows with motion equal to 0, we think they are photos but since there's no "Social permalink" we have no way to verify

## SATURATION

In [ ]:
data["saturation"].head(100)
# video saturation

In [ ]:
data["saturation"].isna().sum()
# no NaN

In [ ]:
data["saturation"].drop_duplicates().sort_values()
data["saturation"].describe()
# no issues

## LUMINANCE

In [ ]:
data["luminance"].head(100)
# video luminance

In [ ]:
data["luminance"].isna().sum()
# no NaN

In [ ]:
data["luminance"].drop_duplicates().sort_values()
data["luminance"].describe()
# no issues

## CONTRAST

In [ ]:
data["contrast"].head(100)
# video contrast

In [ ]:
data["contrast"].isna().sum()
# no NaN

In [ ]:
data["saturation"].drop_duplicates().sort_values()
data["saturation"].describe()
# no issues

## SHARPNESS

In [ ]:
data["sharpness"].head(100)
# video sharpness

In [ ]:
data["sharpness"].isna().sum()
# no NaN

In [ ]:
data["sharpness"].drop_duplicates().sort_values()
data["sharpness"].describe()
# no issues

## COLOR COMPLEXITY

In [ ]:
data["color_complexity"].head(100)
# extent of different colors and color changes between pixels of the video

In [ ]:
data["color_complexity"].isna().sum()
# no NaN

In [ ]:
data["color_complexity"].drop_duplicates().sort_values()
data["color_complexity"].describe()
# no issues

# EXPORT CLEANED DATASET

In [26]:
# export cleaned dataset
data.to_csv("uniting_cleaned.csv", index=False)